# Streams, events and CUDA graphs: timing, overlap and launch overhead

How CUDA orders work with streams, how to overlap copies with compute using pinned memory and several streams, how to express dependencies with events, and how CUDA graphs remove per-launch CPU overhead, first in C and then the way inference engines use them through PyTorch, with static buffers and one graph per batch size.

This notebook holds the code from the lesson **[Streams, events and CUDA graphs: timing, overlap and launch overhead](https://inferenceacademy.com/learn/kern-cuda/streams-events-and-cuda-graphs)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## Streams

Snippet from the lesson (add it to your program; not run on its own):

```cpp
cudaStream_t s1, s2;
cudaStreamCreate(&s1); cudaStreamCreate(&s2);
kernel_a<<<grid, block, 0, s1>>>(...);   // the 4th launch argument is the stream
kernel_b<<<grid, block, 0, s2>>>(...);   // may overlap with kernel_a
```

## Overlapping copies with compute

Snippet from the lesson (add it to your program; not run on its own):

```cpp
const int chunks = 4, n = 1 << 24, cn = n / chunks;
float *h, *d;
cudaMallocHost(&h, n * sizeof(float));          // pinned
cudaMalloc(&d, n * sizeof(float));
cudaStream_t st[chunks];
for (int i = 0; i < chunks; i++) cudaStreamCreate(&st[i]);
for (int i = 0; i < chunks; i++) {
  float* hp = h + i * cn; float* dp = d + i * cn;
  cudaMemcpyAsync(dp, hp, cn * sizeof(float), cudaMemcpyHostToDevice, st[i]);
  process<<<(cn + 255) / 256, 256, 0, st[i]>>>(dp, cn);
  cudaMemcpyAsync(hp, dp, cn * sizeof(float), cudaMemcpyDeviceToHost, st[i]);
}
cudaDeviceSynchronize();
```

## Events: timing and dependencies

Snippet from the lesson (add it to your program; not run on its own):

```cpp
cudaEvent_t done_a;
cudaEventCreateWithFlags(&done_a, cudaEventDisableTiming);   // cheaper when only used for ordering
kernel_a<<<g, b, 0, s1>>>(...);
cudaEventRecord(done_a, s1);
cudaStreamWaitEvent(s2, done_a, 0);   // s2 waits for kernel_a, without blocking the CPU
kernel_b<<<g, b, 0, s2>>>(...);       // runs after kernel_a
```

## CUDA graphs

In [ ]:
%%writefile graph.cu
// graph.cu: measure launch overhead, eager versus graph
#include <cstdio>
__global__ void tiny(float* x) { x[threadIdx.x] += 1.f; }

int main() {
  float* x; cudaMalloc(&x, 32 * sizeof(float));
  cudaStream_t s; cudaStreamCreate(&s);
  cudaEvent_t a, b; cudaEventCreate(&a); cudaEventCreate(&b);
  const int K = 1000; float ms;

  for (int i = 0; i < K; i++) tiny<<<1, 32, 0, s>>>(x);            // warm-up
  cudaEventRecord(a, s);
  for (int i = 0; i < K; i++) tiny<<<1, 32, 0, s>>>(x);
  cudaEventRecord(b, s); cudaEventSynchronize(b);
  cudaEventElapsedTime(&ms, a, b);
  printf("eager: %.2f us per kernel\n", ms * 1000 / K);

  cudaGraph_t g; cudaGraphExec_t ge;
  cudaStreamBeginCapture(s, cudaStreamCaptureModeGlobal);
  for (int i = 0; i < K; i++) tiny<<<1, 32, 0, s>>>(x);            // recorded, not run
  cudaStreamEndCapture(s, &g);
  cudaGraphInstantiate(&ge, g, 0);                                  // CUDA 12 signature
  cudaGraphLaunch(ge, s); cudaStreamSynchronize(s);                 // warm-up
  cudaEventRecord(a, s);
  cudaGraphLaunch(ge, s);
  cudaEventRecord(b, s); cudaEventSynchronize(b);
  cudaEventElapsedTime(&ms, a, b);
  printf("graph: %.2f us per kernel\n", ms * 1000 / K);
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o graph graph.cu && ./graph

### What a graph can't do

## How engines use graphs

In [ ]:
import torch
model = torch.nn.Sequential(*[torch.nn.Linear(1024, 1024) for _ in range(24)]).cuda().half().eval()
static_x = torch.zeros(8, 1024, device="cuda", dtype=torch.half)   # fixed input buffer

side = torch.cuda.Stream()                       # warm up on a side stream, as the docs require
side.wait_stream(torch.cuda.current_stream())
with torch.cuda.stream(side), torch.no_grad():
    for _ in range(3): model(static_x)
torch.cuda.current_stream().wait_stream(side)

g = torch.cuda.CUDAGraph()
with torch.cuda.graph(g), torch.no_grad():
    static_y = model(static_x)                   # recorded; static_y is a fixed output buffer

def run(x):                                      # x must have the captured shape
    static_x.copy_(x)
    g.replay()
    return static_y.clone()                      # clone, or the next replay overwrites it

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-cuda/streams-events-and-cuda-graphs